<a href="https://colab.research.google.com/github/Abdulmuj33b/Wall/blob/main/Final-Assignment-without-dash2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# COVID-19 Global Trends Analysis Report
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
from io import StringIO

# Load and prepare the data
def load_and_prepare_data():
    # Download the dataset directly from Our World in Data
    url = "https://covid.ourworldindata.org/data/owid-covid-data.csv"
    response = requests.get(url)
    data = StringIO(response.text)
    df = pd.read_csv(data)

    # Convert date column to datetime
    df['date'] = pd.to_datetime(df['date'])

    # Calculate derived metrics
    df['cases_per_million'] = df['total_cases'] / (df['population'] / 1e6)
    df['deaths_per_million'] = df['total_deaths'] / (df['population'] / 1e6)
    df['vaccination_rate'] = df['people_vaccinated'] / df['population']

    # Handle missing data
    df['continent'] = df['continent'].fillna('Other')
    df = df.dropna(subset=['population', 'total_cases', 'total_deaths'])

    # Get latest date and create latest data dataframe
    latest_date = df['date'].max()
    latest_df = df[df['date'] == latest_date]

    return df, latest_df, latest_date

# Global overview analysis and visualization
def global_overview_analysis(latest_df):
    # Global totals
    global_totals = latest_df.groupby('continent').agg({
        'total_cases': 'sum',
        'total_deaths': 'sum',
        'people_vaccinated': 'sum',
        'population': 'sum'
    }).reset_index()

    global_totals['case_fatality_rate'] = global_totals['total_deaths'] / global_totals['total_cases'] * 100
    global_totals['vaccination_rate'] = global_totals['people_vaccinated'] / global_totals['population'] * 100

    fig = make_subplots(rows=2, cols=2, subplot_titles=(
        'Total Cases by Continent', 'Total Deaths by Continent',
        'Case Fatality Rate (%)', 'Vaccination Rate (%)'
    ))

    fig.add_trace(
        go.Bar(x=global_totals['continent'], y=global_totals['total_cases']/1e6, name='Cases (M)'),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(x=global_totals['continent'], y=global_totals['total_deaths']/1e6, name='Deaths (M)'),
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(x=global_totals['continent'], y=global_totals['case_fatality_rate'], name='Fatality Rate'),
        row=2, col=1
    )

    fig.add_trace(
        go.Bar(x=global_totals['continent'], y=global_totals['vaccination_rate'], name='Vaccination Rate'),
        row=2, col=2
    )

    fig.update_layout(height=800, title_text="Global COVID-19 Overview by Continent")
    fig.show()

    return global_totals

# Time series analysis and visualization
def time_series_analysis(df):
    # Global daily trends
    global_daily = df.groupby('date').agg({
        'new_cases': 'sum',
        'new_deaths': 'sum',
        'people_vaccinated': 'sum'
    }).reset_index()

    # 7-day rolling averages
    global_daily['new_cases_7day'] = global_daily['new_cases'].rolling(7).mean()
    global_daily['new_deaths_7day'] = global_daily['new_deaths'].rolling(7).mean()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=global_daily['date'],
        y=global_daily['new_cases_7day'],
        name='New Cases (7-day avg)',
        line=dict(color='orange')
    ))

    fig.add_trace(go.Scatter(
        x=global_daily['date'],
        y=global_daily['new_deaths_7day'],
        name='New Deaths (7-day avg)',
        line=dict(color='red')
    ))

    fig.add_trace(go.Scatter(
        x=global_daily['date'],
        y=global_daily['people_vaccinated'].diff().rolling(7).mean()/1e6,
        name='New Vaccinations (7-day avg, M)',
        line=dict(color='green')
    ))

    fig.update_layout(
        title='Global COVID-19 Trends Over Time',
        xaxis_title='Date',
        yaxis_title='Count',
        hovermode='x unified'
    )

    fig.show()

    return global_daily

# Country comparison analysis and visualization
def country_comparison(latest_df):
    # Top 20 countries by cases per capita
    # First drop rows with missing vaccination_rate
    temp_df = latest_df.dropna(subset=['vaccination_rate'])
    top_countries = temp_df.nlargest(20, 'cases_per_million')[['location', 'cases_per_million', 'deaths_per_million', 'vaccination_rate']]

    fig = px.scatter(
        top_countries,
        x='cases_per_million',
        y='deaths_per_million',
        size='vaccination_rate',
        color='location',
        hover_name='location',
        log_x=True,
        log_y=True,
        size_max=60,
        title='COVID-19 Outcomes by Country (Cases vs Deaths per Million)'
    )

    fig.update_layout(xaxis_title='Cases per Million (log scale)',
                     yaxis_title='Deaths per Million (log scale)')
    fig.show()

    return top_countries

# Vaccination progress analysis and visualization
def vaccination_progress(df):
    # Vaccination timeline for selected countries
    countries = ['United States', 'United Kingdom', 'India', 'Brazil', 'South Africa', 'Japan']
    vax_df = df[df['location'].isin(countries)].dropna(subset=['people_vaccinated'])

    fig = px.line(
        vax_df,
        x='date',
        y='vaccination_rate',
        color='location',
        title='Vaccination Rates Over Time by Country'
    )

    fig.update_layout(yaxis_title='Percentage Vaccinated',
                     yaxis=dict(tickformat=".0%"))
    fig.show()

    return vax_df

# Socioeconomic factors analysis and visualization
def socioeconomic_analysis(latest_df):
    # Relationship between GDP and COVID outcomes
    gdp_df = latest_df.dropna(subset=['gdp_per_capita', 'cases_per_million', 'deaths_per_million'])

    fig = px.scatter(
        gdp_df,
        x='gdp_per_capita',
        y='cases_per_million',
        color='continent',
        hover_name='location',
        trendline='lowess',
        title='GDP per Capita vs COVID-19 Cases per Million'
    )

    fig.update_layout(xaxis_title='GDP per Capita (USD)',
                     yaxis_title='Cases per Million')
    fig.show()

    return gdp_df

# Main execution
def main():
    # Load and prepare data
    df, latest_df, latest_date = load_and_prepare_data()
    print(f"Dataset covers from {df['date'].min().date()} to {latest_date.date()}")

    # Run analyses
    global_totals = global_overview_analysis(latest_df)
    global_daily = time_series_analysis(df)
    top_countries = country_comparison(latest_df)
    vax_df = vaccination_progress(df)
    gdp_df = socioeconomic_analysis(latest_df)

    # Export cleaned data
    latest_df.to_csv('covid_cleaned_data.csv', index=False)
    print("Analysis complete. Cleaned data exported to 'covid_cleaned_data.csv'")

if __name__ == "__main__":
    main()

Dataset covers from 2020-01-05 to 2024-08-04


Analysis complete. Cleaned data exported to 'covid_cleaned_data.csv'
